# MiniLibX Documentation Search Engine

A documentation search engine for the **MiniLibX** C graphics library (42paris/minilibx-linux).

## 1. Setup

In [ ]:
%pip install -q 'qdrant-client[fastembed]' openai PyGithub python-slugify python-dotenv pydantic llama-index-core

In [ ]:
import os
import json
import time
import logging
import warnings
from pathlib import Path

from dotenv import load_dotenv
from slugify import slugify
from pydantic import BaseModel
from github import Github, GithubException
from openai import OpenAI
from qdrant_client import QdrantClient, models

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

load_dotenv()

QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]
LLM_BASE_URL = os.environ["LLM_BASE_URL"]
LLM_API_KEY = os.environ["LLM_API_KEY"]
LLM_MODEL = os.environ["LLM_MODEL"]
GITHUB_TOKEN = os.environ["GITHUB_TOKEN"]

GITHUB_REPO = "42Paris/minilibx-linux"
COLLECTION_NAME = "docs_search"
BASE_URL = "/docs/minilibx"
DATA_DIR = Path("data")
GENERATED_DIR = DATA_DIR / "generated"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "Qdrant/bm25"
COLBERT_MODEL = "colbert-ir/colbertv2.0"

print("Setup complete.")
print(f"  Qdrant:  {QDRANT_URL}")
print(f"  LLM:     {LLM_BASE_URL} ({LLM_MODEL})")
print(f"  GitHub:  {GITHUB_REPO}")

## 2. Source Fetching

Fetch MiniLibX source code from GitHub using **PyGithub** with an authenticated token for rate-limit handling.

The library is organized into 9 logical modules covering initialization, window management, image manipulation, drawing primitives, event handling, the main loop, mouse control, XPM image loading, and display synchronization.

In [ ]:
MODULES = {
    "getting-started": ["mlx_init.c"],
    "window": ["mlx_new_window.c", "mlx_clear_window.c",
              "mlx_destroy_window.c", "mlx_int_anti_resize_win.c"],
    "image": ["mlx_new_image.c", "mlx_get_data_addr.c",
             "mlx_put_image_to_window.c", "mlx_destroy_image.c"],
    "drawing": ["mlx_pixel_put.c", "mlx_string_put.c",
              "mlx_set_font.c", "mlx_get_color_value.c"],
    "events": ["mlx_hook.c", "mlx_key_hook.c", "mlx_mouse_hook.c",
              "mlx_expose_hook.c", "mlx_int_param_event.c",
              "mlx_int_set_win_event_mask.c"],
    "loop": ["mlx_loop.c", "mlx_loop_hook.c", "mlx_flush_event.c"],
    "mouse": ["mlx_mouse.c", "mlx_mouse_hook.c"],
    "xpm": ["mlx_xpm.c", "mlx_lib_xpm.c"],
    "sync": ["mlx_destroy_display.c", "mlx_screen_size.c",
            "mlx_int_get_visual.c"],
}

BREADCRUMBS = {
    "getting-started": ["MiniLibX", "Getting Started"],
    "window": ["MiniLibX", "Window"],
    "image": ["MiniLibX", "Image"],
    "drawing": ["MiniLibX", "Drawing"],
    "events": ["MiniLibX", "Events"],
    "loop": ["MiniLibX", "Loop"],
    "mouse": ["MiniLibX", "Mouse"],
    "xpm": ["MiniLibX", "XPM"],
    "sync": ["MiniLibX", "Sync & Display"],
}

In [ ]:
gh = Github(GITHUB_TOKEN)
repo = gh.get_repo(GITHUB_REPO)
print(f"Connected to {repo.full_name} ({repo.description})")


def fetch_file(repository, path):
    try:
        return repository.get_contents(path).decoded_content.decode("utf-8")
    except GithubException as e:
        logger.warning(f"File not found: {path} ({e})")
        return None


def fetch_header(repository):
    return fetch_file(repository, "mlx.h") or ""


def fetch_man_pages(repository):
    try:
        contents = repository.get_contents("man/man3")
        pages = {}
        for f in contents:
            if f.name.endswith(".3"):
                text = fetch_file(repository, f.path)
                if text:
                    pages[f.name] = text
        return pages
    except GithubException as e:
        logger.warning(f"Could not fetch man pages: {e}")
        return {}


def fetch_module_sources(repository, file_list):
    sources = {}
    for filename in file_list:
        content = fetch_file(repository, filename)
        if content:
            sources[filename] = content
    return sources


header = fetch_header(repo)
man_pages = fetch_man_pages(repo)
print(f"Header: {len(header)} chars")
print(f"Man pages: {len(man_pages)} files ({', '.join(list(man_pages.keys())[:5])}...)")

In [ ]:
all_sources = {}
for module_name, file_list in MODULES.items():
    sources = fetch_module_sources(repo, file_list)
    all_sources[module_name] = sources
    print(f"  {module_name}: {len(sources)}/{len(file_list)} files fetched")

## 3. Documentation Generation

Generate structured Markdown documentation for each module using an **OpenAI-compatible LLM** with **Pydantic structured output**.

### Why Pydantic structured output?

Using `client.chat.completions.parse(response_format=PydanticModel)` ensures the LLM always returns valid, well-structured data. The OpenAI SDK automatically converts the Pydantic model to a JSON schema with `strict: true`, sends it to the API, and parses the response into a typed Pydantic instance. No extra library needed.

In [ ]:
llm_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)


class FunctionDoc(BaseModel):
    name: str
    signature: str
    description: str
    parameters: list[dict[str, str]]
    return_value: str
    example: str


class ModuleDoc(BaseModel):
    title: str
    overview: str
    functions: list[FunctionDoc]
    notes: list[str]


SYSTEM_PROMPT = """You are a technical documentation writer for C libraries.
Given source code, a header file, and man pages, produce clear, accurate documentation.

For each function:
- Explain what it does, its parameters, return value, and provide a short usage example.
- Use the actual function signature from the header.
- Describe parameters with name, type, and purpose.

For the module:
- Write a concise overview explaining the module's purpose and how its functions work together.
- Include notes on common pitfalls, thread safety, and platform-specific behavior.
"""

print("LLM client initialized.")
print(f"  Model: {LLM_MODEL}")
print(f"  Base URL: {LLM_BASE_URL}")

In [ ]:
def generate_module_doc(module_name, sources, header_content, man_pages_content):
    cache_path = GENERATED_DIR / f"{module_name}.md"
    if cache_path.exists():
        logger.info(f"Cache hit for {module_name}, skipping generation.")
        return cache_path.read_text()

    source_text = "\n\n".join(
        f"// File: {fname}\n{content}" for fname, content in sources.items()
    )
    man_text = "\n\n".join(
        f"Man page ({fname}):\n{content}" for fname, content in man_pages_content.items()
    )

    user_message = f"""Generate documentation for the "{module_name}" module of MiniLibX.

## Header (mlx.h)
```c
{header_content}
```

## Source Files
```c
{source_text}
```

## Man Pages
{man_text}
"""

    completion = llm_client.chat.completions.parse(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        response_format=ModuleDoc,
    )

    doc: ModuleDoc = completion.choices[0].message.parsed

    md_lines = [f"# {doc.title}\n"]
    md_lines.append(doc.overview + "\n")

    for func in doc.functions:
        md_lines.append(f"## {func.name}\n")
        md_lines.append(func.description + "\n")
        md_lines.append(f"**Signature:** `{func.signature}`\n")
        if func.parameters:
            md_lines.append("**Parameters:**")
            for p in func.parameters:
                md_lines.append(f"- `{p.get('name', '')}` ({p.get('type', '')}): {p.get('description', '')}")
            md_lines.append("")
        md_lines.append(f"**Returns:** {func.return_value}\n")
        md_lines.append(f"**Example:**\n```c\n{func.example}\n```\n")

    if doc.notes:
        md_lines.append("## Notes\n")
        for note in doc.notes:
            md_lines.append(f"- {note}")

    markdown = "\n".join(md_lines)
    cache_path.write_text(markdown)
    logger.info(f"Generated and cached docs for {module_name}.")
    return markdown

In [ ]:
generated_docs = {}
for module_name in MODULES:
    print(f"\nGenerating docs for: {module_name}")
    sources = all_sources.get(module_name, {})
    md = generate_module_doc(module_name, sources, header, man_pages)
    generated_docs[module_name] = md
    print(f"  -> {len(md)} chars")

print(f"\nGenerated docs for {len(generated_docs)} modules.")

## 4. Chunking

Split each module's generated Markdown using a **two-stage LlamaIndex pipeline**: `MarkdownNodeParser` → `SentenceSplitter`.

### Why a two-stage pipeline?

- **Stage 1 — `MarkdownNodeParser`**: Splits by headings, preserving document structure. Each chunk maps to a logical section.
- **Stage 2 — `SentenceSplitter(chunk_size=512, chunk_overlap=50)`**: Sub-splits oversized chunks with sentence-aware boundaries and 10% overlap. This prevents the "giant chunk" problem where a 2800-char section dilutes embeddings.

We also store adjacent context (`prev_section_text`, `next_section_text`) for boundary awareness.

In [ ]:
from llama_index.core.node_parser import MarkdownNodeParser, SentenceSplitter
from llama_index.core import Document

TAG_KEYWORDS = [
    "init", "window", "image", "pixel", "hook", "event", "loop",
    "mouse", "key", "xpm", "destroy", "color", "draw", "sync",
    "expose", "render", "buffer", "display", "error",
]

md_parser = MarkdownNodeParser()
sentence_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)


def extract_tags(module_name, section_title, text):
    tags = {module_name}
    combined = f"{section_title} {text}".lower()
    for kw in TAG_KEYWORDS:
        if kw in combined:
            tags.add(kw)
    return sorted(tags)


def chunk_module(module_name, markdown, breadcrumbs):
    doc = Document(text=markdown, metadata={"module": module_name})
    md_nodes = md_parser.get_nodes_from_documents([doc])

    sub_nodes = sentence_splitter(md_nodes)

    page_slug = slugify(module_name)
    page_url = f"{BASE_URL}/{page_slug}/"

    chunks = []
    for i, node in enumerate(sub_nodes):
        first_line = node.text.strip().split("\n")[0]
        if first_line.startswith("## "):
            section_title = first_line[3:].strip()
        elif first_line.startswith("# "):
            section_title = first_line[2:].strip()
        else:
            section_title = "Overview"

        section_slug = slugify(section_title)
        section_url = f"{page_url}#{section_slug}"
        section_breadcrumbs = breadcrumbs + [section_title.title()]

        prev_text = sub_nodes[i - 1].text[:300] if i > 0 else ""
        next_text = sub_nodes[i + 1].text[:300] if i < len(sub_nodes) - 1 else ""

        chunks.append({
            "page_title": breadcrumbs[-1],
            "section_title": section_title,
            "page_url": page_url,
            "section_url": section_url,
            "breadcrumbs": section_breadcrumbs,
            "chunk_text": node.text.strip(),
            "prev_section_text": prev_text,
            "next_section_text": next_text,
            "tags": extract_tags(module_name, section_title, node.text),
            "module": module_name,
            "chunk_index": i,
        })

    return chunks

In [ ]:
all_chunks = []
for module_name, md in generated_docs.items():
    chunks = chunk_module(module_name, md, BREADCRUMBS[module_name])
    all_chunks.extend(chunks)
    print(f"  {module_name}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")

## 5. Collection Setup & Ingestion

Create a Qdrant collection with **three vector types** and ingest all chunks using the **Document API**.

### Why three separate vector types?

- **Dense** (BAAI/bge-small-en-v1.5, 384-dim): Captures semantic meaning — handles paraphrases and vocabulary mismatch
- **Sparse** (Qdrant/bm25 + IDF modifier): Captures exact keyword matches — handles technical terms, function names
- **ColBERT** (colbert-ir/colbertv2.0, 128-dim): Token-level interaction for precise reranking

### Why ColBERT with m=0?

ColBERT's token-level vectors are too numerous for efficient HNSW indexing. Setting `m=0` disables HNSW, making this field usable **only** for second-stage reranking on candidates from the prefetch stage. This saves RAM and makes the design intent explicit.

In [ ]:
qclient = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print(f"Connected to Qdrant at {QDRANT_URL}")

In [ ]:
existing = [c.name for c in qclient.get_collections().collections]

if COLLECTION_NAME in existing:
    print(f"Collection '{COLLECTION_NAME}' already exists, skipping creation.")
else:
    qclient.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            "dense": models.VectorParams(
                size=384,
                distance=models.Distance.COSINE,
            ),
            "colbert": models.VectorParams(
                size=128,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM,
                ),
                hnsw_config=models.HnswConfigDiff(m=0),
            ),
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                modifier=models.Modifier.IDF,
            ),
        },
    )
    print(f"Collection '{COLLECTION_NAME}' created.")

qclient.create_payload_index(COLLECTION_NAME, "module", models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(COLLECTION_NAME, "tags", models.PayloadSchemaType.KEYWORD)
qclient.create_payload_index(COLLECTION_NAME, "page_title", models.PayloadSchemaType.KEYWORD)
print("Payload indexes created.")

In [ ]:
points = []
for idx, chunk in enumerate(all_chunks):
    text = chunk["chunk_text"]

    points.append(
        models.PointStruct(
            id=idx,
            vector={
                "dense": models.Document(text=text, model=DENSE_MODEL),
                "sparse": models.Document(text=text, model=SPARSE_MODEL),
                "colbert": models.Document(text=text, model=COLBERT_MODEL),
            },
            payload=chunk,
        )
    )

print(f"Built {len(points)} points.")

In [ ]:
BATCH_SIZE = 5
for i in range(0, len(points), BATCH_SIZE):
    batch = points[i:i + BATCH_SIZE]
    qclient.upsert(collection_name=COLLECTION_NAME, points=batch, timeout=120)
    print(f"  Upserted {i + len(batch)}/{len(points)}")

print(f"\nIngestion complete: {len(points)} points in '{COLLECTION_NAME}'.")

## 6. Hybrid Search Pipeline

A two-stage search in a **single `query_points()` call**:

1. **Stage 1**: Dense + Sparse BM25 with RRF fusion → 100 candidates each
2. **Stage 2**: ColBERT MAX_SIM reranking → top 10 results

### Why this pipeline?

Dense vectors capture semantic similarity but miss exact function names. Sparse BM25 captures keywords but misses paraphrases. RRF fusion combines both. ColBERT then reranks with token-level interaction for the final precision boost.

In [ ]:
def search(query, limit=10):
    results = qclient.query_points(
        collection_name=COLLECTION_NAME,
        query=models.Document(text=query, model=COLBERT_MODEL),
        prefetch=[
            models.Prefetch(
                query=models.Document(text=query, model=DENSE_MODEL),
                using="dense",
                limit=100,
            ),
            models.Prefetch(
                query=models.Document(text=query, model=SPARSE_MODEL),
                using="sparse",
                limit=100,
            ),
        ],
        using="colbert",
        with_payload=True,
        limit=limit,
    )
    return results.points

In [ ]:
sample_queries = [
    "how to create a window",
    "put a pixel on screen",
    "handle keyboard events",
]

for q in sample_queries:
    print(f"\nQuery: \"{q}\"")
    results = search(q, limit=3)
    for r in results:
        p = r.payload
        print(f"  [{r.score:.4f}] {p['page_title']} > {p['section_title']}")
        print(f"         {p['section_url']}")

## 7. Result Formatting

Pretty-print search results with page title, section title, URLs, relevance score, and contextual snippets.

In [ ]:
DIVIDER = "\n" + "\u2500" * 60 + "\n"


def format_results(results):
    output_parts = []
    for i, r in enumerate(results, 1):
        p = r.payload
        snippet = p["chunk_text"][:200]
        block = (
            f"{i}. [{r.score:.4f}] {p['page_title']} > {p['section_title']}\n"
            f"   URL: {p['section_url']}\n"
            f"   {snippet}"
        )
        output_parts.append(block)
    return DIVIDER.join(output_parts)


results = search("how to create a window")
print(format_results(results))

## 8. Ground Truth & Evaluation

Evaluate search quality with 25-30 ground truth queries covering four types: **how-to**, **concept**, **API usage**, and **troubleshooting**.

In [ ]:
GROUND_TRUTH = [
    {"query": "how to create a window", "expected_urls": ["window", "mlx_new_window"], "query_type": "how-to"},
    {"query": "put a pixel on screen", "expected_urls": ["drawing", "pixel"], "query_type": "how-to"},
    {"query": "handle keyboard events", "expected_urls": ["events", "key"], "query_type": "how-to"},
    {"query": "load an XPM image", "expected_urls": ["xpm"], "query_type": "how-to"},
    {"query": "destroy a window", "expected_urls": ["window", "destroy"], "query_type": "how-to"},
    {"query": "get mouse position", "expected_urls": ["mouse"], "query_type": "how-to"},
    {"query": "draw text on screen", "expected_urls": ["drawing", "string"], "query_type": "how-to"},
    {"query": "what is MiniLibX", "expected_urls": ["getting-started", "overview", "init"], "query_type": "concept"},
    {"query": "how does the event loop work", "expected_urls": ["loop", "mlx_loop"], "query_type": "concept"},
    {"query": "what are images in MiniLibX", "expected_urls": ["image"], "query_type": "concept"},
    {"query": "how does mlx_hook work", "expected_urls": ["events", "hook"], "query_type": "concept"},
    {"query": "what is the render loop", "expected_urls": ["loop", "loop_hook"], "query_type": "concept"},
    {"query": "how does buffer swapping work", "expected_urls": ["sync", "image", "put_image"], "query_type": "concept"},
    {"query": "mlx_new_window signature and parameters", "expected_urls": ["window", "new_window"], "query_type": "api-usage"},
    {"query": "mlx_get_data_addr usage", "expected_urls": ["image", "data_addr"], "query_type": "api-usage"},
    {"query": "mlx_put_image_to_window arguments", "expected_urls": ["image", "put_image"], "query_type": "api-usage"},
    {"query": "mlx_pixel_put parameters", "expected_urls": ["drawing", "pixel_put"], "query_type": "api-usage"},
    {"query": "mlx_mouse_show or hide cursor", "expected_urls": ["mouse"], "query_type": "api-usage"},
    {"query": "mlx_string_put color and font", "expected_urls": ["drawing", "string_put", "font", "color"], "query_type": "api-usage"},
    {"query": "mlx_destroy_image cleanup", "expected_urls": ["image", "destroy"], "query_type": "api-usage"},
    {"query": "window not showing up", "expected_urls": ["window", "init", "getting-started"], "query_type": "troubleshooting"},
    {"query": "image appears distorted or wrong colors", "expected_urls": ["image", "data_addr", "color", "drawing"], "query_type": "troubleshooting"},
    {"query": "event hook not firing", "expected_urls": ["events", "hook"], "query_type": "troubleshooting"},
    {"query": "segfault when destroying window", "expected_urls": ["window", "destroy", "events"], "query_type": "troubleshooting"},
    {"query": "mlx_loop blocks my program", "expected_urls": ["loop", "mlx_loop", "loop_hook"], "query_type": "troubleshooting"},
    {"query": "XPM file fails to load", "expected_urls": ["xpm"], "query_type": "troubleshooting"},
    {"query": "how to clean up on exit", "expected_urls": ["sync", "destroy", "destroy_display"], "query_type": "how-to"},
    {"query": "get screen resolution", "expected_urls": ["sync", "screen_size"], "query_type": "api-usage"},
]

print(f"Ground truth: {len(GROUND_TRUTH)} queries")
for qt in ["how-to", "concept", "api-usage", "troubleshooting"]:
    count = sum(1 for q in GROUND_TRUTH if q["query_type"] == qt)
    print(f"  {qt}: {count}")

In [ ]:
def url_matches(expected_urls, result_url):
    result_lower = result_url.lower()
    for expected in expected_urls:
        if expected.lower() in result_lower:
            return True
    return False


recall_hits = 0
reciprocal_ranks = []
latencies = []
per_type_results = {}

for gt in GROUND_TRUTH:
    start = time.perf_counter()
    results = search(gt["query"], limit=10)
    elapsed = time.perf_counter() - start
    latencies.append(elapsed)

    found_rank = None
    for rank, r in enumerate(results, 1):
        if url_matches(gt["expected_urls"], r.payload["section_url"]):
            if found_rank is None:
                found_rank = rank
            if found_rank is not None:
                break

    hit = found_rank is not None
    if hit:
        recall_hits += 1
        reciprocal_ranks.append(1.0 / found_rank)
    else:
        reciprocal_ranks.append(0.0)

    qt = gt["query_type"]
    per_type_results.setdefault(qt, {"hits": 0, "total": 0})
    per_type_results[qt]["total"] += 1
    if hit:
        per_type_results[qt]["hits"] += 1

    status = "\u2713" if hit else "\u2717"
    print(f"{status} [{gt['query_type']:15s}] \"{gt['query'][:40]}\" ({elapsed*1000:.0f}ms)")

recall_at_10 = recall_hits / len(GROUND_TRUTH)
mrr = sum(reciprocal_ranks) / len(reciprocal_ranks)
latencies_sorted = sorted(latencies)
p50 = latencies_sorted[int(len(latencies_sorted) * 0.50)]
p95 = latencies_sorted[int(len(latencies_sorted) * 0.95)]

In [ ]:
print("\n" + "=" * 50)
print("EVALUATION SUMMARY")
print("=" * 50)
print(f"Recall@10:  {recall_at_10:.2%}  (target >= 0.80)")
print(f"MRR:        {mrr:.4f}")
print(f"P50 Latency: {p50*1000:.1f} ms")
print(f"P95 Latency: {p95*1000:.1f} ms")
print()
print("Per-Query-Type Recall@10:")
for qt in ["how-to", "concept", "api-usage", "troubleshooting"]:
    data = per_type_results.get(qt, {"hits": 0, "total": 0})
    recall = data["hits"] / data["total"] if data["total"] > 0 else 0
    print(f"  {qt:15s}: {recall:.2%} ({data['hits']}/{data['total']})")
print("=" * 50)

## Summary

This notebook demonstrates a complete documentation search pipeline for the MiniLibX C library:

1. **Source Fetching** — PyGithub with auth token
2. **Doc Generation** — OpenAI-compatible LLM with Pydantic structured output
3. **Chunking** — Section-based with adjacent context, breadcrumbs, and tags
4. **Collection Setup** — Dense + Sparse (BM25+IDF) + ColBERT (m=0) vectors
5. **Ingestion** — Qdrant Document API (zero hand-coded embedding)
6. **Search** — Two-stage hybrid pipeline in a single `query_points()` call
7. **Evaluation** — Recall@10, MRR, P50/P95 latency

### Key Takeaways

- **Separate models** for explicit model embedding type's role
- **Document API** eliminates boilerplate embedding code
- **ColBERT m=0** is intentional — reranking-only, not standalone search
- **BM25 + IDF modifier** ensures proper term-frequency weighting at query time
- **Two-stage search** (RRF fusion → ColBERT rerank) in one API call showcases Qdrant's Universal Query API